# What am I even doing?

None of this makes any sense, so I'm going to make it up as I go along and see what happens, because it's either that or I'm going to [redacted].

Ok.

## What even is $k_i$?

- Everyone has a scalar. I do not have a scalar. This is probably because I am stupid. I need a scalar.
- How do you get a scalar from a time series? You integrate it. So I'm going to integrate from 0 to $t_{ind}$.

$$k_{i} = \int_0^{t_{ind}} \left [ \text{something} \right ] dt$$

- What do I integrate? Rate constants are a stupid idea. I don't care about the constants. I care about what's actually happening, and besides, can you even add forward and reverse rate constants? Probably not. idk. But, net production rate is something that gives us an idea of how much the current reaction is contributing to the chemical makeup of the next time step. I think magnitude is probably more important than direction here (i.e. how much the reaction is contributing to the state change is more important than whether it's forward or backward), so the absolute value of the net reaction rate seems like a good idea.

$$k_{i} = \int_0^{t_{ind}} \left | R_{net} \right | dt$$

- Ok but the rate by itself isn't important; what's important is the _relative_ chemical contribution of the reaction in question, so I probably want to divide by the sum of all of them at a given time step.

$$k_{i} = \int_0^{t_{ind}} \frac{\left | R_{net, i} \right |}{\sum_0^{N} \left | R_{net, j} \right |} dt$$

in units of I don't care what, because I'm going to divide them out anyway (spoilers).

So now I can get the change in integrated relative contribution between the perturbed and unperturbed state for a given equivalence ratio, diluent, and amount of dilution, and relate it to the corresponding change in cell size to get the sensitivity coefficient of the $i$th reaction, $C_{i}$.

$$\Delta k_{i} = k_{i, diluted} - k_{i, undiluted}$$
$$\Delta \lambda = \lambda_{diluted} - k_{undiluted}$$
$$C_{s, i} = \frac{\Delta \lambda}{\Delta k_{i}}$$

Then of course I can divide by the ratio of (unperturbed) cell size to integrated chemical contribution to get the normalized sensitivity coefficient.

$$c_{s, i} = \frac{\Delta \lambda}{\Delta k_{i}} \frac{k_{0, i}}{\lambda_{0}}$$

In [1]:
import seaborn as sns
from matplotlib import pyplot as plt

from scripts.perturbation_study import data

In [2]:
reactions = data.load_reactions()

In [3]:
reactions.undiluted.head()

time  progress  \
condition_id phi_nom reaction                                      
1            1.0     2 CH2 => C2H2 + 2 H  0.000000e+00  0.000000   
                     2 CH2 => C2H2 + 2 H  1.204493e-09  0.001156   
                     2 CH2 => C2H2 + 2 H  2.408987e-09  0.002313   
                     2 CH2 => C2H2 + 2 H  3.613480e-09  0.003469   
                     2 CH2 => C2H2 + 2 H  5.472952e-09  0.005254   

                                          fwd_rate_of_progress  \
condition_id phi_nom reaction                                    
1            1.0     2 CH2 => C2H2 + 2 H          0.000000e+00   
                     2 CH2 => C2H2 + 2 H          3.917624e-12   
                     2 CH2 => C2H2 + 2 H          1.071398e-09   
                     2 CH2 => C2H2 + 2 H          2.025852e-08   
                     2 CH2 => C2H2 + 2 H          2.907649e-07   

                                          rev_rate_of_progress  \
condition_id phi_nom reaction                                    
1            1.0     2 CH2 => C2H2 + 2 H                   0.0   
                     2 CH2 => C2H2 + 2 H                   0.0   
                     2 CH2 => C2H2 + 2 H                   0.0   
                     2 CH2 => C2H2 + 2 H                   0.0   
                     2 CH2 => C2H2 + 2 H                   0.0   

                                          net_rate_of_progress  
condition_id phi_nom reaction                                   
1            1.0     2 CH2 => C2H2 + 2 H          0.000000e+00  
                     2 CH2 => C2H2 + 2 H          3.917624e-12  
                     2 CH2 => C2H2 + 2 H          1.071398e-09  
                     2 CH2 => C2H2 + 2 H          2.025852e-08  
                     2 CH2 => C2H2 + 2 H          2.907649e-07

In [4]:
reactions.diluted.head()

diluent          time  \
condition_id dil_condition phi_nom reaction                                    
327          low           1.0     2 CH2 => C2H2 + 2 H     CO2  0.000000e+00   
                                   2 CH2 => C2H2 + 2 H     CO2  2.036051e-09   
                                   2 CH2 => C2H2 + 2 H     CO2  4.072103e-09   
                                   2 CH2 => C2H2 + 2 H     CO2  6.108154e-09   
                                   2 CH2 => C2H2 + 2 H     CO2  9.528264e-09   

                                                        progress  \
condition_id dil_condition phi_nom reaction                        
327          low           1.0     2 CH2 => C2H2 + 2 H  0.000000   
                                   2 CH2 => C2H2 + 2 H  0.000745   
                                   2 CH2 => C2H2 + 2 H  0.001490   
                                   2 CH2 => C2H2 + 2 H  0.002235   
                                   2 CH2 => C2H2 + 2 H  0.003486   

                                                        fwd_rate_of_progress  \
condition_id dil_condition phi_nom reaction                                    
327          low           1.0     2 CH2 => C2H2 + 2 H          0.000000e+00   
                                   2 CH2 => C2H2 + 2 H          1.358966e-12   
                                   2 CH2 => C2H2 + 2 H          4.528023e-10   
                                   2 CH2 => C2H2 + 2 H          8.462304e-09   
                                   2 CH2 => C2H2 + 2 H          1.298574e-07   

                                                        rev_rate_of_progress  \
condition_id dil_condition phi_nom reaction                                    
327          low           1.0     2 CH2 => C2H2 + 2 H                   0.0   
                                   2 CH2 => C2H2 + 2 H                   0.0   
                                   2 CH2 => C2H2 + 2 H                   0.0   
                                   2 CH2 => C2H2 + 2 H                   0.0   
                                   2 CH2 => C2H2 + 2 H                   0.0   

                                                        net_rate_of_progress  
condition_id dil_condition phi_nom reaction                                   
327          low           1.0     2 CH2 => C2H2 + 2 H          0.000000e+00  
                                   2 CH2 => C2H2 + 2 H          1.358966e-12  
                                   2 CH2 => C2H2 + 2 H          4.528023e-10  
                                   2 CH2 => C2H2 + 2 H          8.462304e-09  
                                   2 CH2 => C2H2 + 2 H          1.298574e-07

In [5]:
conditions = data.load_conditions()

In [6]:
conditions.undiluted.head()

,,equivalence,dil_mf,cell_size,perturbed_rxn
id,phi_nom,,,,
1,1.0,1.0,0.0,0.006102,<NA>
2,1.0,1.0,0.0,0.006102,3
3,1.0,1.0,0.0,0.006102,11
4,1.0,1.0,0.0,0.006102,0
5,1.0,1.0,0.0,0.006103,14


In [7]:
conditions.diluted.head()

,,,diluent,equivalence,dil_mf,cell_size,perturbed_rxn
id,dil_condition,phi_nom,,,,,
327,low,1.0,CO2,1.0,0.1,0.015682,<NA>
328,low,1.0,CO2,1.0,0.1,0.015682,0
329,low,1.0,CO2,1.0,0.1,0.015682,1
330,low,1.0,CO2,1.0,0.1,0.015684,2
331,low,1.0,CO2,1.0,0.1,0.015682,3


In [ ]:
# todo: calculate d(cell_size)/dk_i
#  - LOG PERTURBATION FRACTION
#  - RE-READ SENSITIVITY PAPER -- CAN WE JUST USE PERT FRACTION RATHER THAN TIMESERIES?
# todo: what is a good k_i to use?